In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# 0. Load processed KPIs from Notebook 01
# ============================================================

data_path = "../data/processed/compustat_kpis.csv"
df = pd.read_csv(data_path)

print("Loaded processed KPIs. Shape:", df.shape)
display(df.head())

# Ensure dlrsn is correctly parsed as nullable integer
if "dlrsn" in df.columns:
    df["dlrsn"] = pd.to_numeric(df["dlrsn"], errors="coerce").astype("Int64")

# Recompute bankruptcy_dlrsn for consistency
if "dlrsn" in df.columns:
    df["bankruptcy_dlrsn"] = df["dlrsn"].isin([2, 3]).astype(int)

print("\nBankruptcy label distribution (bankruptcy_dlrsn):")
print(df["bankruptcy_dlrsn"].value_counts(dropna=False))

# ============================================================
# 1. Select ID + KPI columns
# ============================================================

id_cols = [
    "gvkey", "datadate", "fyear", 
    "conm", "tic", "sic", "fyr",
    "dlrsn", "bankruptcy_dlrsn",
]
id_cols = [c for c in id_cols if c in df.columns]

kpi_cols = [
    "roa", "ebitda_margin", "debt_ratio", "total_debt_to_equity",
    "current_ratio", "cash_ratio", "interest_coverage",
    "cfo_margin", "free_cash_flow", "asset_turnover",
    "sales_growth", "asset_growth",
    "book_to_market", "price_to_book",
    "working_capital_to_assets",
    "retained_earnings_to_assets",
]
kpi_cols = [c for c in kpi_cols if c in df.columns]

print("\nKPI columns kept:", len(kpi_cols))
print(kpi_cols)

df = df[id_cols + kpi_cols]
print("Shape after selecting ID + KPIs:", df.shape)

# ============================================================
# 2. Cleaning of KPI columns
# ============================================================

# Replace infinities
df[kpi_cols] = df[kpi_cols].replace([np.inf, -np.inf], np.nan)

# Drop rows with all KPI = NaN
df = df.dropna(subset=kpi_cols, how="all")
print("After dropping all-NaN KPI rows:", df.shape)

# Remove rows with >50% KPI missing
row_missing_ratio = df[kpi_cols].isna().mean(axis=1)
df = df[row_missing_ratio <= 0.5]
print("After removing rows with >50% KPI missing:", df.shape)

# ============================================================
# 3. Fill missing values (per KPI median)
# ============================================================

for col in kpi_cols:
    df[col] = df[col].fillna(df[col].median())

print("Remaining NaN values (should be 0):", df[kpi_cols].isna().sum().sum())

# ============================================================
# 4. Winsorization (1% - 99%)
# ============================================================

for col in kpi_cols:
    low, high = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(lower=low, upper=high)

print("Winsorization complete.")

# ============================================================
# 5. Save CLEAN (unscaled) version
# ============================================================

output_clean = "../data/processed/compustat_kpis_clean.csv"
df.to_csv(output_clean, index=False)
print("Saved clean dataset:", output_clean)

# ============================================================
# 6. Standardize KPI columns (Z-score)
# ============================================================

df_scaled = df.copy()

for col in kpi_cols:
    mean, std = df_scaled[col].mean(), df_scaled[col].std()
    df_scaled[col] = 0 if (std == 0 or np.isnan(std)) else (df_scaled[col] - mean) / std

print("Standardization complete.")

# ============================================================
# 7. Save SCALED version
# ============================================================

output_scaled = "../data/processed/compustat_kpis_clean_scaled.csv"
df_scaled.to_csv(output_scaled, index=False)
print("Saved scaled dataset:", output_scaled)

# ============================================================
# 8. Summary statistics (only KPIs)
# ============================================================

summary = df[kpi_cols].describe().T
summary_path = "../data/processed/kpi_summary_stats.csv"
summary.to_csv(summary_path)
print("Summary statistics saved to:", summary_path)

print("\nFinal shapes:")
print("Unscaled:", df.shape)
print("Scaled:", df_scaled.shape)
display(summary.head())


Loaded processed KPIs. Shape: (332675, 57)


,gvkey,datadate,fyear,conm,tic,act,lct,at,lt,seq,...,interest_coverage,cfo_margin,free_cash_flow,asset_turnover,sales_growth,asset_growth,working_capital_to_assets,retained_earnings_to_assets,book_to_market,price_to_book
0,1004,1995-05-31,1994,AAR CORP,AIR,321.632,73.140,425.814,228.695,197.119,...,2.242018,0.033795,6.182,1.060076,NaN,NaN,0.583569,0.240565,NaN,1.234818
1,1004,1996-05-31,1995,AAR CORP,AIR,338.012,79.385,437.846,233.211,204.635,...,3.055953,0.049031,17.213,1.153351,0.118732,0.028256,0.590680,0.250182,NaN,1.729691
2,1004,1997-05-31,1996,AAR CORP,AIR,414.100,99.981,529.584,260.325,269.259,...,3.976451,0.016173,-20.761,1.112813,0.167009,0.209521,0.593143,0.231646,NaN,2.095840
3,1004,1998-05-31,1997,AAR CORP,AIR,468.400,149.148,670.559,369.709,300.850,...,4.465020,0.029181,5.328,1.166375,0.327144,0.266200,0.476098,0.220100,NaN,2.434527
4,1004,1999-05-31,1998,AAR CORP,AIR,508.186,173.586,726.630,400.595,326.035,...,4.167663,0.031072,-7.606,1.263416,0.173774,0.083618,0.460482,0.245524,0.602903,1.658646



Bankruptcy label distribution (bankruptcy_dlrsn):
bankruptcy_dlrsn
0    320551
1     12124
Name: count, dtype: int64

KPI columns kept: 16
['roa', 'ebitda_margin', 'debt_ratio', 'total_debt_to_equity', 'current_ratio', 'cash_ratio', 'interest_coverage', 'cfo_margin', 'free_cash_flow', 'asset_turnover', 'sales_growth', 'asset_growth', 'book_to_market', 'price_to_book', 'working_capital_to_assets', 'retained_earnings_to_assets']
Shape after selecting ID + KPIs: (332675, 23)
After dropping all-NaN KPI rows: (276316, 23)
After removing rows with >50% KPI missing: (240308, 23)
Remaining NaN values (should be 0): 0
Winsorization complete.
Saved clean dataset: ../data/processed/compustat_kpis_clean.csv
Standardization complete.
Saved scaled dataset: ../data/processed/compustat_kpis_clean_scaled.csv
Summary statistics saved to: ../data/processed/kpi_summary_stats.csv

Final shapes:
Unscaled: (240308, 23)
Scaled: (240308, 23)


,count,mean,std,min,25%,50%,75%,max
roa,240308.0,-0.294836,1.444858,-11.815924,-0.071920,0.017381,0.061630,0.348705
ebitda_margin,240308.0,-1.356236,8.377378,-71.087312,0.015394,0.113829,0.235843,0.784319
debt_ratio,240308.0,0.918641,2.100383,0.028815,0.351549,0.590874,0.848984,18.431481
total_debt_to_equity,240308.0,0.791244,2.873902,-11.025820,0.001957,0.335757,1.057264,17.362048
current_ratio,240308.0,2.672375,3.470963,0.012904,1.156315,1.719343,2.632603,23.958693
